# Construirea aplicațiilor de generare a imaginilor (Azure OpenAI)

LLM-urile nu se limitează doar la generarea de text. Poți genera și imagini din descrieri textuale. Imaginile, ca modalitate, sunt utile în domenii precum MedTech, arhitectură, turism, dezvoltarea jocurilor, marketing și altele. În această lecție lucrăm cu modelele **GPT Image** actuale pe Microsoft Foundry.

## Obiective de învățare

Până la finalul acestei lecții vei putea să:

- Explici ce este generarea de imagini și unde este utilă.
- Înțelegi familia de modele `gpt-image` și cum se diferențiază de modelele DALL·E vechi.
- Construiești o aplicație de generare de imagini și să editezi imagini.

## Ce este generarea de imagini?

Modelele de generare a imaginilor creează imagini pornind de la un prompt text. Modelele moderne, precum `gpt-image`, învață relația dintre text și imagini în timpul antrenării, apoi transformă iterativ zgomotul aleator într-o imagine care corespunde promptului tău.

Două familii cunoscute de modele pentru imagini sunt:

- **`gpt-image` (OpenAI)** — generația curentă folosită în această lecție. Suportă generarea text-în-imagine și editarea imaginilor (inpainting cu o mască).
- **Midjourney** — un model popular third-party cu propriul serviciu și un flux de lucru bazat pe Discord.

> Modelele OpenAI mai vechi pentru imagini — **DALL·E 2** și **DALL·E 3** — sunt de domeniul trecutului. DALL·E 3 nu mai este disponibil pentru noi implementări, iar funcții precum `create_variation` au existat doar în DALL·E 2. Folosește modelele `gpt-image` pentru aplicațiile noi.

Pe Microsoft Foundry, **`gpt-image-2`** este cel mai recent și cel mai capabil model de imagini și este recomandat ca implicit. `gpt-image-1.5` și `gpt-image-1-mini` sunt de asemenea disponibile în general.

> **Important:** modelele `gpt-image` returnează imaginea generată ca **base64** (`b64_json`), nu ca URL. Codul tău decodează șirul base64 în bytes și îl salvează — nu există URL de imagine pentru descărcare.


## Construirea primei tale aplicații de generare de imagini

Deci ce este necesar pentru a construi o aplicație de generare de imagini? Ai nevoie de următoarele biblioteci:

- **python-dotenv**, este foarte recomandat să folosești această bibliotecă pentru a-ți păstra secretele într-un fișier *.env* separat de cod.
- **openai**, această bibliotecă este cea pe care o vei folosi pentru a interacționa cu API-ul OpenAI.
- **pillow**, pentru a lucra cu imagini în Python.

Dacă nu ai făcut-o deja, urmează instrucțiunile de pe pagina [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) pentru a crea o resursă și un model Microsoft Foundry. Selectează **gpt-image-2** ca model (cel mai nou model Azure OpenAI pentru imagini; DALL·E este învechit).

1. Creează un fișier *.env* cu următorul conținut:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Găsește aceste informații în portalul Microsoft Foundry pentru resursa ta, în secțiunea "Deployments".



1. Colectați bibliotecile de mai sus într-un fișier numit *requirements.txt* astfel:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Următorul pas, creați un mediu virtual și instalați bibliotecile:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pentru Windows, folosiți următoarele comenzi pentru a crea și activa mediul virtual:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Adăugați următorul cod în fișierul numit *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # configurează clientul Azure OpenAI (Microsoft Foundry).
    # Modelele de imagine necesită o versiune recentă a API-ului — verifică documentația Foundry pentru versiunea necesară modelului tău.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Creează o imagine folosind API-ul de generare a imaginilor
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introdu textul solicitării tale aici
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # de ex. "gpt-image-2"
        )
        # Setează directorul pentru imaginea stocată
        image_dir = os.path.join(os.curdir, 'images')

        # Dacă directorul nu există, creează-l
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inițializează calea imaginii (atenție, tipul fișierului trebuie să fie png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # modelele gpt-image returnează imaginea ca base64 (b64_json), nu un URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Afișează imaginea în vizualizatorul implicit de imagini
        image = Image.open(image_path)
        image.show()

    # prinde excepțiile
    except BadRequestError as err:
        print(err)

    ```

Hai să explicăm acest cod:

- Mai întâi, importăm bibliotecile de care avem nevoie, inclusiv biblioteca OpenAI, biblioteca dotenv, modulul base64 și biblioteca Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Apoi, încărcăm variabilele de mediu din fișierul *.env*.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- După aceea, configurăm clientul Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Următorul pas este să generăm imaginea:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Introduceți aici textul promptului dvs.
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modelele `gpt-image` returnează imaginea ca un șir **base64** în `data[0].b64_json`. Îl decodăm în octeți și îl scriem într-un fișier — nu există un URL de unde să îl descărcăm.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- În final, deschidem imaginea și folosim vizualizatorul standard de imagini pentru a o afișa:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mai multe detalii despre generarea imaginii

Să analizăm parametrii funcției `images.generate`:

- **prompt**, este textul folosit pentru a genera imaginea. Aici este „Iepuraș pe cal, ținând un acadea, pe un pajiște cețoasă unde cresc narcise”.
- **size**, este dimensiunea imaginii generate (de exemplu `1024x1024`, `1536x1024`, `1024x1536` sau `"auto"`).
- **n**, este numărul de imagini generate. Aici generăm una.
- **model**, este numele implementării modelului de imagine (de exemplu `gpt-image-2`).

> Modelele de imagine nu acceptă parametrul `temperature` — acesta este un control pentru generarea de text. Pentru varietate, apelează API-ul din nou; pentru a reduce varietatea, fă promptul mai specific.

## Capabilități suplimentare ale generării de imagini

Ați văzut cum se generează o imagine cu câteva linii de cod Python. Modelele `gpt-image` pot **edita** și o imagine existentă. Furnizând o imagine, o **mască** opțională (care marchează zona ce trebuie schimbată) și un prompt, puteți modifica o parte a imaginii — de exemplu, să adăugați o pălărie iepurașului nostru.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# editările sunt, de asemenea, returnate ca base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Imaginea de bază conține doar iepurele; imaginea finală are pălăria pe iepure.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
